### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="blood_tests_drink_prediction",
    dataset_year="1996",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C54G67",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/60/liver+disorders.zip &&  unzip liver+disorders.zip bupa.data && rm liver+disorders.zip
mkdir -p local-data-warehouse/blood_tests_drink_prediction && mv bupa.data local-data-warehouse/blood_tests_drink_prediction/
""",
    # References
    academic_reference_bibtex=r"""@misc{UCILiverDisorders2016,
  title        = {Liver Disorders},
  author       = {{UCI Machine Learning Repository}},
  year         = {2016},
  howpublished = {\url{https://doi.org/10.24432/C54G67}},
  note         = {Dataset}
}
""",
    academic_reference_bibtex_key="UCILiverDisorders2016",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
The task is framed as a liver disorder prediction task, but we only have data bout the amount of drinks consumed. So instead, we will frame it as a task to predict the number of drinks based on the blood work data. This is a proxy for the original task, where later based on the number of drinks the liver disorder was determined.

- We drop the selector column, as we ignore the original train/test split, and will create our own splits.
- The data contains natural duplicates, so we keep them.
- We log1p scale the target.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="drinks",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv(dataset_mold.path / "bupa.data", header=None, names=["mcv", "alkphos", "sgpt", "sgot", "gammagt", "drinks", "selector"])
print("Loaded data shape:", df.shape)

df = df.drop(columns=["selector"])
df["drinks"] = np.log1p(df["drinks"])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()